In [ ]:
%pip install dbx-tools-graphiti==0.9.54

In [ ]:
import json
import uuid
from datetime import datetime, timezone
from typing import Any

from dbx_tools.graphiti.runtime import GraphitiRuntime
from graphiti_core.nodes import EpisodeType, EpisodicNode

"""Install Graphiti from PyPI, then verify persistence across fresh runtimes."""

dbutils: Any = globals()["dbutils"]

DATABASE_URL = "projects/lfp-chat-db/branches/production/endpoints/primary"
MARKER = f"dbx-tools-pypi-demo-{uuid.uuid4()}"


async def _write_episode() -> str:
    """Write one episode through a new GraphitiRuntime."""
    runtime = GraphitiRuntime.from_options({"databaseUrl": DATABASE_URL})
    await runtime.start()
    try:
        added = await runtime.graphiti.add_episode(
            name="dbx-tools PyPI runtime demo",
            episode_body=f"Persistent marker: {MARKER}",
            source_description="dbx-tools-graphiti PyPI demo",
            source=EpisodeType.text,
            group_id="main",
            reference_time=datetime.now(timezone.utc),
        )
        return str(added.episode.uuid)
    finally:
        await runtime.close()


async def _read_episode(episode_uuid: str) -> dict[str, object]:
    """Read the episode through a separate GraphitiRuntime."""
    runtime = GraphitiRuntime.from_options({"databaseUrl": DATABASE_URL})
    await runtime.start()
    try:
        episode = await EpisodicNode.get_by_uuid(runtime.graphiti.driver, episode_uuid)
        if MARKER not in episode.content:
            raise AssertionError(
                f"Episode {episode_uuid} did not contain the expected marker {MARKER}"
            )
        return {
            "episode_uuid": str(episode.uuid),
            "found": True,
            "marker": MARKER,
            "matched_content": episode.content,
        }
    finally:
        await runtime.close()


episode_uuid = await _write_episode()
result = await _read_episode(episode_uuid)
dbutils.notebook.exit(json.dumps(result, sort_keys=True))